<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 CrewAI 自定义工具:`@tool` vs `BaseTool`

**一句话定位**:`@tool` 是函数式定义,`BaseTool` 是面向对象定义,两者都能创建 Custom Tool,区别在于要不要管理状态/配置/继承。

</div>

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

在 CrewAI 中,`@tool` 装饰器和继承 `BaseTool` 都可以创建自定义工具(Custom Tool)。两者都能完成大部分常见的 Tool 功能,但 **`BaseTool` 更适合需要复杂配置、状态管理和自定义行为**的工具。

官方参考:[CrewAI — Create Custom Tools](https://docs.crewai.com/en/learn/create-custom-tools)

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 1️⃣ 两种方式的代码对比

</div>

假设我们需要创建一个计算器工具。

**方式一:使用 `@tool`(简单、推荐优先使用)**

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
from crewai.tools import tool

@tool("Calculator")
def calculator(a: int, b: int) -> int:
    """Add two integers together."""
    return a + b
</code></pre>

CrewAI 会自动从函数中获取:

- `name`:`Calculator`
- `description`:函数的 docstring
- `args_schema`:根据参数类型 `a: int, b: int` 自动生成
- 执行逻辑:函数体

不需要显式定义 Pydantic Schema。

**方式二:继承 `BaseTool`(更灵活)**

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
from crewai.tools import BaseTool
from pydantic import BaseModel, Field

class CalculatorInput(BaseModel):
    a: int = Field(description="First integer")
    b: int = Field(description="Second integer")

class CalculatorTool(BaseTool):
    name: str = "Calculator"
    description: str = "Add two integers together."
    args_schema: type[BaseModel] = CalculatorInput

    def _run(self, a: int, b: int) -> int:
        return a + b
</code></pre>

这里需要:

1. 创建 Pydantic 输入模型
2. 创建继承 `BaseTool` 的类
3. 定义 `name`、`description` 和 `args_schema`
4. 实现 `_run()` 方法

代码明显更多,但可以进一步自定义工具行为。两种方式都符合 CrewAI 官方文档。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 2️⃣ 功能对比

</div>

| 比较维度 | `@tool` | `BaseTool` |
|---|---|---|
| 代码量 | 少 | 多 |
| 定义方式 | Function | Class |
| 参数 Schema | 自动生成 | 通常显式定义,也可推断 |
| Pydantic 验证 | 支持 | 支持 |
| 异步执行 | 支持 `async def` | 支持 `_arun()` |
| 自定义缓存 | 支持 | 支持 |
| 结构化返回值 | 支持 | 支持 |
| 状态和依赖管理 | 相对不方便 | 更自然 |
| 继承与方法重写 | 不适合 | 支持 |
| 高级输出格式控制 | 有限 | 可重写相关方法 |
| 典型场景 | 简单、独立的工具 | 复杂、可复用的工具 |

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**关键洞察**:不是说 `@tool` 无法实现复杂逻辑,而是当复杂逻辑需要**管理状态**或**扩展框架行为**时,`BaseTool` 通常更清晰。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 3️⃣ `BaseTool` 真正有什么优势?

</div>

**优势一:管理工具自身的状态与配置**

例如,一个工具可能需要保存 API 配置、客户端对象或其他依赖。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
from crewai.tools import BaseTool

class SearchTool(BaseTool):
    name: str = "Search"
    description: str = "Search information."
    api_key: str
    max_results: int = 10

    def _run(self, query: str) -> str:
        return (
            f"Searching '{query}' "
            f"with max_results=<span style=\"background:#3d3414; color:#fde047; padding:0 2px;\">{self.max_results}</span>"
        )

search_tool = SearchTool(
    api_key="your-api-key",
    max_results=5
)
</code></pre>

这里的 `api_key` 和 `max_results` 都是**实例配置**,而不是 Agent 每次调用工具时传入的参数。当工具需要维护较多配置时,Class 的组织方式更合适。

**优势二:可以自定义框架行为**

CrewAI 允许 `BaseTool` 子类重写 `format_output_for_agent()`,控制工具执行结果以什么文本形式传递给 Agent。这可以将复杂的内部返回对象转换成更精简的 Agent 可读结果,而不改变直接调用工具时得到的原始结果。

相比之下,`@tool` 更侧重于快速定义工具的执行逻辑。

**优势三:更适合较大型项目**

假设你正在开发一个 Agent,需要集成:

- GitHub API
- PostgreSQL
- Slack API

每个工具都有自己的身份认证、配置、客户端和错误处理逻辑。这时可以将这些工具封装成独立的 Class,方便进行依赖管理、单元测试和复用。

<div class="dark-orange" style="background:#2d2418; color:#fed7aa; padding:10px 24px; border-left:4px solid #fb923c; border-radius:4px; width:97%;"><style>.dark-orange strong{color:#67e8f9;}</style>

💡 不过,这属于软件工程上的优势,而不是 CrewAI 强制要求使用 `BaseTool`。

</div>

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 4️⃣ 两种方式最终有什么区别?

</div>

从 Agent 使用工具的角度来看,通常**没有本质区别**。

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
from crewai import Agent

agent = Agent(
    role="Math Assistant",
    goal="Solve math problems",
    backstory="An expert in mathematics.",
    tools=[calculator]  # @tool version
)
</code></pre>

或者:

<pre style="background:#1e1e2e; color:#d4d4d4; padding:10px 24px; border-radius:4px; font-size:0.90em; width:97%;">
<code class="language-python">
agent = Agent(
    role="Math Assistant",
    goal="Solve math problems",
    backstory="An expert in mathematics.",
    tools=[CalculatorTool()]  # BaseTool version
)
</code></pre>

两种方式都把一个可调用的 Tool 交给 Agent。只要工具名称、描述、参数及执行逻辑相同,Agent 的使用体验通常也相同。

<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:16px 28px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

### 5️⃣ 应该选择哪一种?

</div>

**默认选择 `@tool`**:简单 API 调用、数学计算、文本处理、数据库查询等无复杂内部状态的工具。

**需要封装复杂行为时选择 `BaseTool`**:需要管理多个配置项、客户端对象、内部状态,或需要通过继承重写框架方法的工具。

你可以将它们理解为 Python 中两种常见的设计选择:

- `@tool`:用一个函数完成一个明确的任务。
- `BaseTool`:用一个对象封装功能、配置、状态和可扩展行为。

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**:学习 CrewAI 时优先使用 `@tool`,理解 `BaseTool` 的基本结构(特别是 `args_schema` 和 `_run`)即可;等到实际项目需要更复杂的工具封装(状态、配置、继承、自定义输出格式)时,再切换到 `BaseTool`。

官方参考:[CrewAI — Create Custom Tools](https://docs.crewai.com/en/learn/create-custom-tools)

</div>